## Exploratory data analysis: Beijing air quality

This notebook loads daily station records from the MySQL table `air_quality.aqi_daily` and explores pollutant concentrations alongside daily ERA5 weather summaries. It summarizes station differences, daily and monthly patterns, and pairwise associations in the available records.

The analysis covers PM2.5, PM10, SO2, NO2, CO, O3, and nine weather features. The plots and statistics are calculated from the table each time the notebook runs; they reflect its current date range, station coverage, and missing values. These descriptive analyses show patterns and associations, not causal effects.

## 1. Import libraries

Load the libraries used for data handling, plotting, and database access.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

plt.style.use("seaborn-v0_8-whitegrid")

## 2. Connect to the database

Connect to MySQL and read the daily station records from `air_quality.aqi_daily`. The notebook requires the database and table to be available before running.

In [ ]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")

    df = pd.read_sql_query("SELECT * FROM aqi_daily", engine)
    combined_df = df.copy()
    print(f"Loaded {len(df):,} daily station rows from aqi_daily.")
except Exception as error:
    raise RuntimeError(
        "Could not connect to MySQL or load the aqi_daily table. "
        "Check that MySQL is running and air_quality.aqi_daily exists."
    ) from error

In [ ]:
combined_df.head()


In [ ]:
missing_values = combined_df.isnull().sum()

print(missing_values)

## 3. Select pollutants and weather features

Select the six pollutant summaries and nine daily weather features used below. Values are converted to numeric where possible; missing or invalid values become null and are omitted from the relevant pandas summaries and correlations.

In [ ]:
pollutants = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
]

weather_measurements = [
    "TEMP_mean",
    "TEMP_range",
    "PRES_mean",
    "DEWP_mean",
    "RH_mean",
    "RAIN_sum",
    "WSPM_mean",
    "wd_sin",
    "wd_cos",
]

analysis_columns = pollutants + weather_measurements
missing_columns = [column for column in analysis_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"aqi_daily is missing analysis columns: {missing_columns}")
df[analysis_columns] = df[analysis_columns].apply(pd.to_numeric, errors="coerce")

print("Pollutants included in the analysis:")
print(pollutants)
print("Daily weather features included in the analysis:")
print(weather_measurements)

pollutant_statistics = df[pollutants].describe()
pollutant_statistics

## 4. Station averages for each pollutant

Calculate the mean concentration of each pollutant by station and plot each pollutant separately. Separate axes make station comparisons readable because pollutants use different units and scales. A station mean summarizes the available daily records; unequal coverage or missing readings can affect comparisons.

In [ ]:
station_averages = combined_df.groupby("station")[pollutants].mean()

station_averages

for pollutant in pollutants:

    ax = station_averages[pollutant].plot(
        kind="bar",
        figsize=(14, 6)
    )

    ax.set_title(
        f"Average {pollutant} Concentration by Monitoring Station"
    )

    ax.set_xlabel("Monitoring Station")
    ax.set_ylabel("Average Concentration")
    ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

In [ ]:
for pollutant in pollutants:

    highest_station = station_averages[pollutant].idxmax()
    highest_value = station_averages[pollutant].max()

    lowest_station = station_averages[pollutant].idxmin()
    lowest_value = station_averages[pollutant].min()

    overall_average = station_averages[pollutant].mean()

    print(f"\n{pollutant}")
    print("=" * 50)

    print(
        f"Overall average concentration : "
        f"{overall_average:.2f}"
    )

    print(
        f"Highest average station       : "
        f"{highest_station}"
    )

    print(
        f"Highest average concentration : "
        f"{highest_value:.2f}"
    )

    print(
        f"Lowest average station        : "
        f"{lowest_station}"
    )

    print(
        f"Lowest average concentration  : "
        f"{lowest_value:.2f}"
    )

### How to read these charts

Each bar is the mean of the available daily values for one station and pollutant. The printed highest and lowest station values are descriptive summaries for the loaded data, not evidence that station location alone explains the difference.

## 5. Daily pollutant averages across stations

For each date, calculate the mean across station records available for that pollutant, then plot the resulting daily series. This gives a citywide view of day-to-day variation, while the number of contributing stations may vary by date.

In [ ]:
combined_df["date"] = pd.to_datetime(combined_df["date"], errors="coerce")
if combined_df["date"].isna().any():
    raise ValueError("aqi_daily contains invalid dates.")

daily_pollutants = (
    combined_df
    .groupby("date")[pollutants]
    .mean()
)

daily_pollutants.head()

In [ ]:
for pollutant in pollutants:
    plt.figure(figsize=(15, 6))
    
    plt.plot(
        daily_pollutants.index,
        daily_pollutants[pollutant]
    )
    
    plt.title(f"Daily Average {pollutant} Across 12 Monitoring Stations")
    plt.xlabel("Date")
    plt.ylabel("Average Concentration")
    plt.tight_layout()
    plt.show()

In [ ]:
for pollutant in pollutants:

    daily_station = (
        combined_df
        .groupby(["date", "station"])[pollutants]
        .mean()
    )

    daily_values = daily_station[pollutant].dropna()

    highest_value = daily_values.max()
    highest_date, highest_station = daily_values.idxmax()

    lowest_value = daily_values.min()
    lowest_date, lowest_station = daily_values.idxmin()

    print(f"\n{pollutant}")
    print("=" * 50)

    print(
        f"Highest daily concentration : {highest_value:.2f}"
    )
    print(
        f"Station with highest value  : {highest_station}"
    )
    print(
        f"Date of highest value       : {highest_date.date()}"
    )

    print(
        f"Lowest daily concentration  : {lowest_value:.2f}"
    )
    print(
        f"Station with lowest value   : {lowest_station}"
    )
    print(
        f"Date of lowest value        : {lowest_date.date()}"
    )

### How to read these trends

The series shows observed fluctuations and peaks in the station-averaged daily values. Weather, season, emissions, station coverage, and other factors may contribute to these patterns; this plot alone cannot identify their causes.

## 6. Monthly pollutant averages across stations

Group the daily station records by calendar month and calculate pollutant means. Monthly aggregation smooths day-to-day variation and helps inspect broad seasonal patterns. Months may contain different numbers of observations or stations.

In [ ]:
monthly_pollutants = (
    combined_df
    .groupby(combined_df["date"].dt.to_period("M"))[pollutants]
    .mean()
)

monthly_pollutants.index = monthly_pollutants.index.astype(str)

monthly_pollutants.head()

In [ ]:
for pollutant in pollutants:
    plt.figure(figsize=(15, 6))

    plt.plot(
        monthly_pollutants.index,
        monthly_pollutants[pollutant],
        marker="o",
        linewidth=2
    )

    plt.title(f"Monthly Average {pollutant} Across 12 Monitoring Stations")
    plt.xlabel("Month")
    plt.ylabel("Average Concentration")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
for pollutant in pollutants:

    monthly_station = (
        combined_df
        .groupby([combined_df["date"].dt.to_period("M"), "station"])[pollutants]
        .mean()
    )

    monthly_values = monthly_station[pollutant].dropna()

    highest_value = monthly_values.max()
    highest_month, highest_station = monthly_values.idxmax()

    lowest_value = monthly_values.min()
    lowest_month, lowest_station = monthly_values.idxmin()

    print(f"\n{pollutant}")
    print("=" * 50)

    print(
        f"Highest monthly concentration : {highest_value:.2f}"
    )
    print(
        f"Station with highest value    : {highest_station}"
    )
    print(
        f"Month of highest value       : {highest_month.to_timestamp().strftime('%Y-%m')}"
    )

    print(
        f"Lowest monthly concentration  : {lowest_value:.2f}"
    )
    print(
        f"Station with lowest value     : {lowest_station}"
    )
    print(
        f"Month of lowest value        : {lowest_month.to_timestamp().strftime('%Y-%m')}"
    )

### How to read these trends

Compare recurring rises and falls across months, keeping in mind that the chart pools available station records and does not adjust for changes in coverage or other factors. Seasonal patterns are descriptive and do not establish a cause.

## Pollutant and weather associations: Pearson correlation

Calculate Pearson correlations across pollutant and weather measurements in the daily station records. Pearson correlation summarizes linear association and can be affected by outliers, seasonality, and repeated observations over time.

In [ ]:
pearson_corr = df[analysis_columns].corr(method="pearson")
print(pearson_corr.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(
    pearson_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title("Pollutants and Weather Measurements")
plt.tight_layout()
plt.show()

### Interpretation

Values near +1 indicate positive linear association, values near -1 indicate negative linear association, and values near 0 indicate little linear association. Correlation does not establish causation; shared seasonal patterns or station differences may contribute.

## Pollutant and weather associations: Spearman correlation

Calculate Spearman rank correlations for the same measurements. This summarizes monotonic association using ranks and can capture patterns that are not linear, though it also does not establish causation.

In [ ]:
spearman_corr = df[analysis_columns].corr(method="spearman")
print(spearman_corr.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(
    spearman_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title("Ranked Relationships Among Pollutants and Weather Measurements")
plt.tight_layout()
plt.show()

### Interpretation

Compare the rank-based results with Pearson correlations to see whether associations are similar under the two measures. Differences can reflect nonlinearity, outliers, or the distribution of the data; neither measure identifies a causal relationship.

## Strongest pairwise Pearson correlations

Rank the unique pollutant and weather pairs by the absolute value of their Pearson correlation and display the ten largest. The chart retains the sign so positive and negative associations remain distinguishable.

In [ ]:
upper_triangle = np.triu_indices_from(pearson_corr, k=1)
strongest_pairs = pd.DataFrame({
    "variable_1": pearson_corr.index[upper_triangle[0]],
    "variable_2": pearson_corr.columns[upper_triangle[1]],
    "correlation": pearson_corr.to_numpy()[upper_triangle],
})
strongest_pairs["absolute_correlation"] = strongest_pairs["correlation"].abs()
strongest_pairs = strongest_pairs.sort_values(
    "absolute_correlation", ascending=False
).head(10)
strongest_pairs["relationship"] = (
    strongest_pairs["variable_1"] + " and " + strongest_pairs["variable_2"]
)
print(
    strongest_pairs[["variable_1", "variable_2", "correlation"]]
    .round(3)
    .to_string(index=False)
)

plt.figure(figsize=(10, 6))
colors = [
    "steelblue" if value >= 0 else "indianred"
    for value in strongest_pairs["correlation"]
]
plt.barh(
    strongest_pairs["relationship"][::-1],
    strongest_pairs["correlation"][::-1],
    color=colors[::-1],
)
plt.axvline(0, color="black", linewidth=0.8)
for index, value in enumerate(strongest_pairs["correlation"][::-1]):
    offset = 0.02 if value >= 0 else -0.02
    alignment = "left" if value >= 0 else "right"
    plt.text(
        value + offset,
        index,
        f"{value:.2f}",
        va="center",
        ha=alignment,
    )
plt.xlabel("Pearson correlation")
plt.title("Strongest Relationships")
plt.xlim(-1, 1)
plt.tight_layout()
plt.show()

### Interpretation

The displayed pairs are the largest absolute linear correlations in the currently loaded data. Their ordering and values can change with date range, station coverage, and missing data. They are descriptive associations, not causal effects.

## PM2.5 correlations with other pollutants by station

Calculate within-station Pearson correlations between PM2.5 and each of the other five pollutants. Comparing stations can show whether these pairwise linear associations vary across locations.

In [ ]:
station_corr = (
    df.groupby("station")[pollutants]
    .corr()["PM2.5"]
    .unstack(level=1)
    .drop(columns="PM2.5")
)
print(station_corr.round(3))

plt.figure(figsize=(9, 7))
sns.heatmap(
    station_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
)
plt.title("PM2.5 Relationships with Other Pollutants by Station")
plt.tight_layout()
plt.show()

### Interpretation

Read each row as the within-station correlation between PM2.5 and another pollutant. Differences across stations describe variation in association in this dataset; they do not show that one pollutant causes changes in another. Sample size and missingness can also affect the estimates.